Migrated from Databricks to Neuron (auto-port).

In [ ]:
import os
import sys
from pathlib import Path

for _candidate in (
    "/home/jovyan/work/default/rashmi-ranjan/lib",
    str(Path.cwd().parent / "lib"),
    str(Path.cwd() / "lib"),
):
    if _candidate not in sys.path and Path(_candidate).is_dir():
        sys.path.insert(0, _candidate)

from neuron_runtime import (
    GoogleSheetReader,
    SlackReportsTrigger,
    get_sql_df,
    load_service_account,
    load_slack_token,
    make_unique_columns,
)

service_account = load_service_account()
slack_token = load_slack_token()

In [ ]:
import os
import shutil

# Step 1: Ensure 'data' directory exists
if not os.path.exists("data"):
    os.makedirs("data")

# Step 2: Run the SQL query and get the results as a DataFrame
query = """
WITH t AS (
    SELECT 
        CAST(json_extract(dt.metadata, '$.correlation_id') AS STRING) AS chat_reference_id,
        result,
        us.name AS annotatedBy,
        rs.annotated_at AS annotatedAt,
        dt.ticket_id AS ticket_id
    FROM (
        SELECT id, data_id, annotated_by, annotated_at, result, status
        FROM annotation.annotation_results
        WHERE DATE(from_utc_timestamp(CAST(SPLIT_PART(REPLACE(annotated_at, 'T', ' '), '.', 1) AS TIMESTAMP), 'Asia/Kolkata'))
              BETWEEN DATE_ADD(CURRENT_DATE(), -90) AND CURRENT_DATE()
    ) rs
    LEFT JOIN annotation.annotation_data dt
        ON rs.data_id = dt.id
    LEFT JOIN mjolnir_service.users us
        ON us.account_id = rs.annotated_by
    WHERE dt.template_id = 177
),

parsed AS (
    SELECT
        chat_reference_id,
        annotatedAt,
        annotatedBy,
        result,
        CAST(from_json(result, 'array<struct<from_name:string,value:struct<choices:array<string>,text:string>>>') AS ARRAY<STRUCT<from_name:STRING,value:STRUCT<choices:ARRAY<STRING>,text:STRING>>>) AS x
    FROM t
),

data AS (
    SELECT
        chat_reference_id,
        annotatedBy,
        CAST(from_utc_timestamp(TO_TIMESTAMP(SUBSTRING(REPLACE(annotatedAt,'T',' '),1,19),'yyyy-MM-dd HH:mm:ss'),'Asia/Kolkata') AS DATE) AS annotated_at_date,
        from_utc_timestamp(TO_TIMESTAMP(SUBSTRING(REPLACE(annotatedAt,'T',' '),1,19),'yyyy-MM-dd HH:mm:ss'),'Asia/Kolkata') AS annotated_at_ist,

        MAX(CASE WHEN x.from_name = 'category' THEN to_json(x.value.choices) END) AS `Category`,
        MAX(CASE WHEN x.from_name LIKE '%subcategory%' THEN to_json(x.value.choices) END) AS `Sub-Category`,
        MAX(CASE WHEN x.from_name = 'correctint' THEN to_json(x.value.choices) END) AS `Correct interpretation of customer's query / message`,
        MAX(CASE WHEN x.from_name = 'clarification' THEN to_json(x.value.choices) END) AS `Asked for customer clarification for any response that it could not understand`,
        MAX(CASE WHEN x.from_name = 'Othertxn' THEN to_json(x.value.choices) END) AS `Reference to correct LAN / transaction ID`,
        MAX(CASE WHEN x.from_name = 'correctinfo' THEN to_json(x.value.choices) END) AS `Correct information as per SOP / guidelines`,
        MAX(CASE WHEN x.from_name = 'hallucination' THEN to_json(x.value.choices) END) AS `No hallucinations (by GenAI Bot)`,
        MAX(CASE WHEN x.from_name = 'unnecessary' THEN to_json(x.value.choices) END) AS `No Unnecessary information`,
        MAX(CASE WHEN x.from_name = 'latch' THEN to_json(x.value.choices) END) AS `Not latching to VOC`,
        MAX(CASE WHEN x.from_name = 'allqueries' THEN to_json(x.value.choices) END) AS `Addressed all queries of the customer`,
        MAX(CASE WHEN x.from_name = 'allinfo' THEN to_json(x.value.choices) END) AS `Given all information regarding a query(e.g. conveying 3-5 days TAT instead of saying soon)`,
        MAX(CASE WHEN x.from_name = 'empathy' THEN to_json(x.value.choices) END) AS `Demonstrated understanding of customer emotions and concerns`,
        MAX(CASE WHEN x.from_name = 'easyunder' THEN to_json(x.value.choices) END) AS `Replies are easy to understand (no technical terms / jargons)`,
        MAX(CASE WHEN x.from_name = 'language' THEN to_json(x.value.choices) END) AS `Adherence to customer language preference`,
        MAX(CASE WHEN x.from_name = 'allmessage' THEN to_json(x.value.choices) END) AS `Did GENAI consider all back-to back customer messages and not just last message?`,
        MAX(CASE WHEN x.from_name = 'loop' THEN to_json(x.value.choices) END) AS `Did not go on a loop when customer said Thanks / Ok / any other acknowledgement`,
        MAX(CASE WHEN x.from_name = 'upload' THEN to_json(x.value.choices) END) AS `Was it able to comprehend the uploaded document?`,
        MAX(CASE WHEN x.from_name = 'profess' THEN to_json(x.value.choices) END) AS `Professional behaviour with customer(not referring to customers with slangs like bhai, yaar etc.)`,
        MAX(CASE WHEN x.from_name = 'openclose' THEN to_json(x.value.choices) END) AS `Opening / Closing`,
        MAX(CASE WHEN x.from_name = 'cxend' THEN to_json(x.value.choices) END) AS `Did customer end chat after we provided resolution?`,
        MAX(CASE WHEN x.from_name = 'rep' THEN to_json(x.value.choices) END) AS `Unique responses by the agent (No word-to-word repititions)`,
        MAX(CASE WHEN x.from_name = 'closure' THEN to_json(x.value.choices) END) AS `Did GENAI close the ticket? (If customer abandoned, select No)`,
        MAX(CASE WHEN x.from_name = 'csat' THEN to_json(x.value.choices) END) AS `Closing script adherence (Correct CSAT script)`,
        MAX(CASE WHEN x.from_name = 'escalationagent' THEN to_json(x.value.choices) END) AS `Correctly escalated to an agent`,
        MAX(CASE WHEN x.from_name = 'GENAI' THEN to_json(x.value.choices) END) AS `Was GENAI Disposition correct?`,
        MAX(CASE WHEN x.from_name = 'remarks' THEN to_json(x.value.text) END) AS `Remarks`
    FROM parsed
    LATERAL VIEW EXPLODE(x) t AS x
    GROUP BY chat_reference_id, annotatedBy, annotated_at_date, annotated_at_ist
)

SELECT DISTINCT a.*, b.csat
FROM data a
LEFT JOIN (
    SELECT canonical_ticket_id, AVG(csat) AS csat
    FROM corp_analytics.cx_sub_ticket_data_model_tbl
    WHERE ticket_created_dt >= DATE_SUB(CURRENT_DATE(), 90)
    GROUP BY canonical_ticket_id
) b
ON a.chat_reference_id = b.canonical_ticket_id
ORDER BY a.annotated_at_date DESC;
"""
df = get_sql_df(query)

# Step 3: Connect to the Google Sheet
gsheet_url = "https://docs.google.com/spreadsheets/d/1OSjWnb9eOSmHLHZF3yKGg9hIT-YJlgihJv6mMJpg__Y/edit?gid=0#gid=0"
worksheet_name = "CL-GenAI"
gs_reader = GoogleSheetReader(gsheet_url, service_account)

# Step 4: Clear existing data from the worksheet
gs_reader.clear_sheets(worksheet_name, start_row=1, start_column=1)

# Step 5: Write the new DataFrame to the worksheet (including headers)
gs_reader.set_df_in_sheets(worksheet_name, df, include_headers=True, row=1, col=1)